# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pratheekrao12/-flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

I chose **Lane 2 — Refresh / Content Opportunity Scoring**. I am interested in turning visible content and search-performance signals into a practical review queue, rather than treating every page as equally urgent.

The starter dataset is useful because it contains one row per pseudonymized content page, plus observable freshness, search visibility, traffic, and trend signals. This lane fits the internship's goal of an evidence-backed decision: it can compare pages and propose which ones deserve attention first. It does not require claiming that a refresh will cause growth.

In [1]:
lane = "Lane 2 — Refresh / Content Opportunity Scoring"
print(lane)

assert lane.startswith("Lane 2")

Lane 2 — Refresh / Content Opportunity Scoring


## 2. The question: decision, action, cost of a wrong call

**Research question:** “Can observable content and search-performance signals help identify and prioritize web pages that should be reviewed for possible content improvement?”

- **Unit of analysis:** one pseudonymized page (`content_id`) in the starter dataset snapshot; client membership may be used only for grouping.
- **Decision supported:** which pages should enter a review queue and in what order.
- **Expected output:** a ranked page-level score, a short list of reason codes, a suggested review action, and a confidence or uncertainty label where the later analysis supports one.
- **Who acts:** a FlyRank SEO/content reviewer, editor, or account strategist uses the queue; they make the final judgment.
- **Possible actions:** refresh outdated copy, expand thin or incomplete sections, protect a healthy page near Page 1, monitor a page with weak evidence, merge/remove redundant coverage, or leave a page unchanged.

**False positive:** the system recommends a review that later evidence shows was unnecessary. The cost is wasted editor/strategist time and delay for truly higher-value pages.

**False negative:** the system does not prioritize a page that would benefit from review or misses an emerging problem. The cost is missed improvement opportunity, delayed action, and possible lost traffic or ranking opportunity.

The intended output is a decision-support ranking for review. It is not a promise that editing a page will improve traffic.

In [2]:
unit_of_analysis = "one pseudonymized content page"
expected_output = "ranked review queue with a score, suggested action, and reason codes"
decision = "which pages should be reviewed first for possible content improvement"
actor = "SEO/content reviewer or editor"

print({
    "unit_of_analysis": unit_of_analysis,
    "decision": decision,
    "actor": actor,
    "expected_output": expected_output,
})

{'unit_of_analysis': 'one pseudonymized content page', 'decision': 'which pages should be reviewed first for possible content improvement', 'actor': 'SEO/content reviewer or editor', 'expected_output': 'ranked review queue with a score, suggested action, and reason codes'}


## 3. Quick look at the data (2–3 real numbers)

The starter dataset has **30,000 pages** and **44 columns**. It provides a single snapshot with trailing 90-day search and analytics measures.

The measured trend distribution is **16,262 down, 4,388 up, 5,962 stable, 2,236 new, and 1,152 flat**. Therefore, **16,262 pages (54.21%) are declining** by the dataset's `trend_direction` rule.

Search volume and 90-day impressions have a Pearson correlation of **0.0012** (using 27,532 rows with nonblank search-volume values). This is effectively no linear association in the snapshot, not evidence of a universal relationship between the variables or proof of performance cause.

Finally, **17 pages are both stale and visible** under the explicit review-rule thresholds: `days_since_last_update >= 180` and `impressions_90d >= 500`. These are the currently identifiable stale-visible candidates, while the score and queue developed later may include other signals.

In [3]:
from pathlib import Path
import pandas as pd

# Locate the starter CSV whether the notebook runs from the repository root,
# the notebook directory, or a parent directory containing the nested repository.
cwd = Path.cwd()
candidates = []
for root in [cwd, *cwd.parents]:
    candidates.append(root / "data" / "raw" / "content_refresh_anonymized.csv")
    candidates.extend(root.glob("*/data/raw/content_refresh_anonymized.csv"))

csv_path = next((p.resolve() for p in candidates if p.is_file()), None)
if csv_path is None:
    raise FileNotFoundError("Could not locate data/raw/content_refresh_anonymized.csv")

pages = pd.read_csv(csv_path)

trend_counts = pages["trend_direction"].value_counts(dropna=False)
trend_counts = trend_counts.reindex(["down", "up", "stable", "new", "flat"], fill_value=0)
total_pages = len(pages)
declining_pages = int((pages["trend_direction"] == "down").sum())
declining_pct = declining_pages / total_pages * 100
volume_impression_correlation = pages["search_volume"].corr(pages["impressions_90d"])
search_volume_pairs = pages[["search_volume", "impressions_90d"]].dropna().shape[0]
stale_visible_pages = int(((pages["days_since_last_update"] >= 180) & (pages["impressions_90d"] >= 500)).sum())

print(f"CSV: {csv_path}")
print(f"Dataset shape: {pages.shape[0]} pages × {pages.shape[1]} columns")
print("Trend distribution:")
for trend, count in trend_counts.items():
    print(f"  {trend}: {int(count)}")
print(f"Declining pages: {declining_pages} ({declining_pct:.2f}%)")
print(f"Correlation (search_volume vs impressions_90d): {volume_impression_correlation:.4f}")
print(f"Pairwise complete correlation rows: {search_volume_pairs}")
print(f"Stale and visible pages: {stale_visible_pages}")

CSV: /Users/pratheekrao/FlyRank_internship/-flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
Dataset shape: 30000 pages × 44 columns
Trend distribution:
  down: 16262
  up: 4388
  stable: 5962
  new: 2236
  flat: 1152
Declining pages: 16262 (54.21%)
Correlation (search_volume vs impressions_90d): 0.0012
Pairwise complete correlation rows: 27532
Stale and visible pages: 17


## 4. Careful words: what I can and can't claim

**What I can claim from this work:**

- I can **describe and measure observations** in the starter snapshot, such as trend counts, the declining share, stale-visible pages, and associations between columns.
- I can build a **directional, decision-support ranking** if transparent rules or later models consistently help reviewers find pages worth manual examination in validation data.
- I can report **predictive performance only after defining an observed outcome window and validating on held-out or future data**. With the starter CSV, any later target must be observed rather than assumed to be a refresh effect.

**What I cannot claim:**

- This dataset does **not prove Google's ranking algorithm**, identify a unique cause of a trend, or show that refresh timing caused impressions to move.
- A refresh recommendation is not a guarantee of higher traffic, rankings, clicks, engagement, or revenue.
- The correlation result is only a snapshot association. The small threshold-based stale-visible count is descriptive, not a demonstrated lift estimate.
- I will avoid causal words such as “refresh caused growth” and performative claims such as “the model knows Google's behavior.” I will use careful language: observed, measured, associated, directional, review-priority, and decision-support.

The analysis may fail for missing data, measurement-window differences, changing search intent, external events, and pages that do not have enough impressions or history. Its honest use is to help a human decide what to inspect next, not to replace editorial judgment or establish causality.

In [4]:
claim_levels = {
    "observed": ["trend counts", "declining percentage", "stale-visible count", "correlation"],
    "directional_or_decision_support": ["ranked review queue", "suggested action", "reason codes"],
    "not_claimed": ["refresh guarantees growth", "Google ranking mechanism", "causal effect"],
}

for level, examples in claim_levels.items():
    print(f"{level}: {', '.join(examples)}")

observed: trend counts, declining percentage, stale-visible count, correlation
directional_or_decision_support: ranked review queue, suggested action, reason codes
not_claimed: refresh guarantees growth, Google ranking mechanism, causal effect


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No private client names, private URLs, or private queries are included
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.